# Evaporating Universe — Paper I
## NB07: N-Body Validation — Gadget-4 EU (Null Test + EU Production)

**Purpose:** Cross-validation of the Gadget-4 EU simulation outputs.
Loads pre-computed P(k) from Null test (no SELFGRAVITY) and EU production,
performs comprehensive checks, and exports `NB07_results.json`.

| Item | Value |
|:-----|:------|
| Simulation | Gadget-4 + EU patch (CDM drain via `eu_tables.h`) |
| Particles | 1024³ = 1,073,741,824 |
| Box size | 500 Mpc/h |
| IC generator | monofonIC (2LPT, z_start = 49) |
| Snapshots | 15 (z = 49, 6.99, 6.02, 5.50, 5.02, 4.50, 4.01, 3.51, 2.99, 2.50, 2.00, 1.50, 1.00, 0.50, 0.00) |
| Null | No SELFGRAVITY — null test (drain only, no structure formation) |
| Prod | Full TreePM — production run (drain + gravity) |
| P(k) extraction | Pylians3 (512 grid, MAS='CIC', compensated) |
| Platform | AWS m8a.metal-48xl (192 vCPU, 768 GB RAM) |

---

## §1. Setup & Data Loading

Parse all 30 P(k) files (15 Null + 15 Production). Extract redshift, scale factor,
and CDM mass from each file header. Build master catalog.

In [ ]:
# §1. SETUP & DATA LOADING
import numpy as np
import matplotlib.pyplot as plt
import json, os, re, glob
from scipy import interpolate
from scipy.integrate import solve_ivp

plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 6),
                     'axes.grid': True, 'grid.alpha': 0.3})

# ── Paths ──
# On Colab: upload all 31 files from NB07/INPUT/ when prompted
# Locally: files are already in the INPUT folder
IS_COLAB = os.path.exists('/content')

if IS_COLAB:
    # Check if files were already uploaded
    if not os.path.exists('NB01_params.json'):
        from google.colab import files
        print('[UPLOAD] Select all 31 files from NB07/INPUT/:')
        print('  - NB01_params.json')
        print('  - 15× pk_NNN_zX.XX.txt  (production)')
        print('  - 15× null_pk_NNN_zX.XX.txt  (null test)')
        uploaded = files.upload()
        print(f'[OK] Uploaded {len(uploaded)} files')
    BASE = '/content'
    # On Colab, all files are flat in /content/
    PROD_DIR = BASE
    NULL_DIR = BASE
    RESULTS_DIR = BASE
else:
    BASE = os.path.dirname(os.path.abspath('__file__'))
    PROD_DIR = os.path.join(BASE, 'NB07', 'INPUT')
    NULL_DIR = os.path.join(BASE, 'NB07', 'INPUT')
    RESULTS_DIR = os.path.join(BASE, 'results')

print(f'[OK] Running on {"Colab" if IS_COLAB else "local"}')
print(f'  Prod dir: {PROD_DIR}')
print(f'  Null dir: {NULL_DIR}')


In [ ]:
# §1b. PARSE P(k) FILES

def parse_pk_file(filepath):
    """Parse a Pylians3 P(k) file.
    Header format: # z=X a=Y mass=Z
    Data columns: k [h/Mpc], P(k) [(Mpc/h)^3], Nmodes
    Returns dict with metadata + arrays."""
    with open(filepath, 'r') as f:
        line1 = f.readline().strip()
        line2 = f.readline().strip()  # column header

    # Parse header: # z=49.0000 a=0.020000 mass=0.9645418902
    m = re.match(r'#\s*z=([\d.]+)\s+a=([\d.]+)\s+mass=([\d.]+)', line1)
    assert m, f'Failed to parse header: {line1} in {filepath}'
    z = float(m.group(1))
    a = float(m.group(2))
    mass = float(m.group(3))

    # Load data
    data = np.loadtxt(filepath, comments='#')
    k_hmpc = data[:, 0]      # h/Mpc
    Pk_mpch3 = data[:, 1]    # (Mpc/h)^3
    nmodes = data[:, 2].astype(int)

    return {
        'z': z, 'a': a, 'mass': mass,
        'k_hmpc': k_hmpc, 'Pk_mpch3': Pk_mpch3, 'nmodes': nmodes,
        'filename': os.path.basename(filepath)
    }


def load_all_pk(directory, pattern, label):
    """Load P(k) files matching pattern, sorted by snapshot number."""
    full_pattern = os.path.join(directory, pattern)
    files = sorted(glob.glob(full_pattern))
    assert len(files) == 15, f'{label}: Expected 15 P(k) files, found {len(files)} matching {pattern}'
    snapshots = []
    for f in files:
        snap = parse_pk_file(f)
        # Extract snapshot number from filename
        # Handles both pk_000_z49.00.txt and null_pk_000_z49.00.txt
        basename = os.path.basename(f)
        parts = basename.replace('null_', '').split('_')
        snap_num = int(parts[1])  # pk_000_... → 000
        snap['snap_id'] = snap_num
        snapshots.append(snap)
    print(f'  {label}: loaded {len(snapshots)} snapshots')
    return snapshots


# Load all data
print('=== LOADING P(k) DATA ===')
prod_snaps = load_all_pk(PROD_DIR, 'pk_[0-9]*.txt', 'Production')
null_snaps = load_all_pk(NULL_DIR, 'null_pk_*.txt', 'Null test')
print(f'\n[OK] Total: {len(prod_snaps)} production + {len(null_snaps)} null test = {len(prod_snaps)+len(null_snaps)} P(k) files')


In [ ]:
# §1c. LOAD GROUND TRUTH FROM NB01

nb01_path = os.path.join(RESULTS_DIR, 'NB01_params.json')
if not os.path.isfile(nb01_path):
    # Colab: try current dir
    nb01_path = 'NB01_params.json'
assert os.path.isfile(nb01_path), f'NB01_params.json not found: {nb01_path}'

with open(nb01_path, 'r') as f:
    nb01 = json.load(f)

# Extract EU parameters (nested under 'eu_derived')
eu = nb01['eu_derived']
eps_IR  = eu['eps_IR']['value']
z_trans = eu['z_trans']['value']
b_exp   = eu['b']['value']
lam     = eu['lambda']['value']

print('=== NB01 GROUND TRUTH ===')
print(f'  ε_IR    = {eps_IR}')
print(f'  z_trans = {z_trans}')
print(f'  b       = {b_exp}')
print(f'  λ       = {lam}')

# Simulation parameters (MCMC C2 + generate_eu_tables.py values)
H0_EU = 68.886  # km/s/Mpc (generate_eu_tables.py line 27)
h_EU = H0_EU / 100.0
omega_cdm_prim = 0.1193  # primordial (generate_eu_tables.py line 29)
omega_b = 0.02237
BOX_MPC_H = 500.0
NPART = 1024**3

# f_cdm(z=0) from CLASS-EU C2 exact value
# (generate_eu_tables.py line 73: calibrated to 0.955765)
fcdm_z0_exact = 0.955765

print(f'  H0_EU   = {H0_EU} km/s/Mpc')
print(f'  h_EU    = {h_EU}')
print(f'  f_cdm(z=0) = {fcdm_z0_exact}')
print(f'  Box     = {BOX_MPC_H} Mpc/h')
print(f'  Npart   = {NPART:,}')


In [ ]:
# §1d. MASTER CATALOG — Summary table

print('=== SNAPSHOT CATALOG ===')
print(f'{"Snap":>4} {"z_prod":>8} {"a_prod":>8} {"mass_prod":>12} {"z_null":>8} {"mass_null":>12} {"Δmass":>10}')
print('-' * 75)

for s3, s2 in zip(prod_snaps, null_snaps):
    dm = abs(s3['mass'] - s2['mass'])
    print(f'{s3["snap_id"]:4d} {s3["z"]:8.2f} {s3["a"]:8.6f} {s3["mass"]:12.10f} '
          f'{s2["z"]:8.2f} {s2["mass"]:12.10f} {dm:10.2e}')

print(f'\nProduction mass(z=49) = {prod_snaps[0]["mass"]:.10f}')
print(f'Production mass(z=0)  = {prod_snaps[-1]["mass"]:.10f}')
print(f'Null test mass(z=49) = {null_snaps[0]["mass"]:.10f}')
print(f'Null test mass(z=0)  = {null_snaps[-1]["mass"]:.10f}')
print(f'\nNB01 f_cdm(z=0) = {fcdm_z0_exact}')

# Quick validation: Null test and Production mass at z=0 should match
delta_mass_z0 = abs(prod_snaps[-1]['mass'] - null_snaps[-1]['mass'])
print(f'\n|mass_prod - mass_null| at z=0 = {delta_mass_z0:.2e}')
assert delta_mass_z0 < 0.001, f'Null/Prod mass mismatch at z=0: {delta_mass_z0}'
print('[OK] §1 complete — all 30 P(k) files loaded and cataloged')

---

## §2. Null Test: Background Drain Validation

Null test was run with `SELFGRAVITY` disabled in Gadget-4. This means:
- **No gravitational forces** → particles don't cluster → P(k) should be flat (white noise at IC amplitude level)
- **CDM drain IS active** → particle masses decrease according to `eu_tables.h`

This is the **null test**: it proves the drain is purely thermodynamic
(background-level mass rescaling), with zero coupling to gravitational dynamics.

### Checks:
1. P(k) is approximately constant across k at each z (no structure)
2. Mass decreases monotonically
3. Mass is constant for z > z_trans (Heaviside cutoff)
4. Total drain matches NB01 analytical prediction

In [ ]:
# §2a. Null test P(k) FLATNESS CHECK
# Without gravity, P(k) should be flat (shot noise / IC-level white noise)
# The amplitude scales as mass^2 (P(k) ∝ m_particle^2 for white noise)

print('=== Null test P(k) FLATNESS CHECK ===')
print(f'{"Snap":>4} {"z":>8} {"P(k) mean":>14} {"P(k) std":>14} {"CV=std/mean":>12} {"Flat?":>6}')
print('-' * 65)

v2_flatness = []
for snap in null_snaps:
    # Use modes with Nmodes > 50 (avoid low-k cosmic variance)
    mask = snap['nmodes'] > 50
    pk_vals = snap['Pk_mpch3'][mask]
    mean_pk = np.mean(pk_vals)
    std_pk = np.std(pk_vals)
    cv = std_pk / mean_pk  # Coefficient of variation
    is_flat = cv < 0.5  # Generous threshold for shot noise
    v2_flatness.append({'z': snap['z'], 'mean': mean_pk, 'std': std_pk, 'cv': cv, 'flat': is_flat})
    status = '✅' if is_flat else '❌'
    print(f'{snap["snap_id"]:4d} {snap["z"]:8.2f} {mean_pk:14.4e} {std_pk:14.4e} {cv:12.4f} {status:>6}')

all_flat = all(f['flat'] for f in v2_flatness)
print(f'\nAll snapshots flat: {"✅ YES" if all_flat else "❌ NO"}')
print('[OK] §2a Null test flatness check complete')

In [ ]:
# §2b. NULL TEST MASS EVOLUTION + DRAIN QUANTIFICATION

z_null = np.array([s['z'] for s in null_snaps])
mass_null = np.array([s['mass'] for s in null_snaps])

# ── Analytical f_cdm(z) from generate_eu_tables.py ──
# Ground truth: sigmoid s(z) = 1/(1 + ((1+z)/(1+z_trans))^(1/b))
# then f_cdm(z) = 1 - eps_eff * s(z) for z <= z_trans, else 1.0
# Calibrated so f_cdm(0) = 0.955765 (CLASS-EU C2)

def sigmoid_s(z, z_t, b):
    """Activation function s(z) per NB01 §3.
    s(z) = 1 / (1 + ((1+z)/(1+z_trans))^(1/b))"""
    ratio = (1.0 + z) / (1.0 + z_t)
    return 1.0 / (1.0 + ratio**(1.0 / b))

# Calibrate epsilon_eff from f_cdm(z=0) = 0.955765
s0 = sigmoid_s(0.0, z_trans, b_exp)
eps_eff = (1.0 - fcdm_z0_exact) / s0
print(f'Calibration: s(0) = {s0:.6f}, eps_eff = {eps_eff:.6f}')

def compute_fcdm(z_arr, z_t, b_val):
    """Compute f_cdm(z) using the EXACT same formula as generate_eu_tables.py.
    Heaviside cutoff: f_cdm = 1.0 for z > z_trans."""
    fcdm = np.ones(len(z_arr))
    for i, z in enumerate(z_arr):
        if z <= z_t:
            fcdm[i] = 1.0 - eps_eff * sigmoid_s(z, z_t, b_val)
    return fcdm

# Verify calibration
fcdm_z0 = compute_fcdm(np.array([0.0]),
                       z_trans, b_exp)[0]
print(f'f_cdm(z=0) computed  = {fcdm_z0:.6f}')
print(f'f_cdm(z=0) expected  = {fcdm_z0_exact}')
assert abs(fcdm_z0 - fcdm_z0_exact) < 1e-6, \
    f'f_cdm calibration failed: {fcdm_z0} vs {fcdm_z0_exact}'

# Compute analytical curve
z_fine = np.linspace(0, 49, 500)
fcdm_fine = compute_fcdm(z_fine, z_trans, b_exp)

# Normalize simulation mass to match analytical convention
fcdm_at_z49 = compute_fcdm(np.array([49.0]),
                           z_trans, b_exp)[0]
mass_null_norm = mass_null / mass_null[0] * fcdm_at_z49

# Compute analytical f_cdm at Null test snapshot redshifts
fcdm_at_snaps = compute_fcdm(z_null, z_trans, b_exp)

print('\n=== NULL TEST DRAIN VALIDATION ===')
print(f'{"Snap":>4} {"z":>8} {"mass_sim":>12} '
      f'{"f_cdm_ana":>12} {"Δ(%)":>10}')
print('-' * 50)
for i, snap in enumerate(null_snaps):
    delta = (mass_null_norm[i] - fcdm_at_snaps[i]) / fcdm_at_snaps[i] * 100
    print(f'{snap["snap_id"]:4d} {snap["z"]:8.2f} '
          f'{mass_null_norm[i]:12.8f} {fcdm_at_snaps[i]:12.8f} '
          f'{delta:+10.4f}%')

# Total drain
mass_above_zt = mass_null[0]
mass_z0 = mass_null[-1]
drain_total_v2 = (1.0 - mass_z0 / mass_above_zt) * 100
drain_analytical = (1.0 - fcdm_z0 / fcdm_at_z49) * 100

print(f'\nTotal drain (Null test sim):  {drain_total_v2:.4f}%')
print(f'Total drain (NB01):    {drain_analytical:.4f}%')
print(f'Difference:            '
      f'{abs(drain_total_v2 - drain_analytical):.4f}%')

assert abs(drain_total_v2 - drain_analytical) < 0.5, \
    f'Drain mismatch: sim={drain_total_v2:.4f}% '\
    f'vs ana={drain_analytical:.4f}%'
print('\n[OK] §2b Null test drain validated against NB01')


In [ ]:
# §2c. NULL TEST VISUALIZATION

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Null test P(k) at selected redshifts (should be flat)
ax = axes[0]
colors_z = plt.cm.viridis(np.linspace(0, 1, 15))
for i, snap in enumerate(null_snaps):
    mask = snap['nmodes'] > 10
    ax.loglog(snap['k_hmpc'][mask], snap['Pk_mpch3'][mask],
              color=colors_z[i], alpha=0.7, lw=0.8,
              label=f'z={snap["z"]:.1f}' if i % 3 == 0 else '')
ax.set_xlabel('k [h/Mpc]')
ax.set_ylabel('P(k) [(Mpc/h)³]')
ax.set_title('Null test: P(k) — No Gravity (Null Test)')
ax.legend(fontsize=8, loc='lower left')

# Panel B: Mass evolution (Null test)
ax = axes[1]
ax.plot(z_null, mass_null, 'bo-', ms=6, lw=2, label='Null test (simulation)')
ax.plot(z_fine, fcdm_fine / fcdm_at_z49 * mass_null[0], 'r--', lw=1.5,
        label='NB01 analytical', alpha=0.8)
ax.axvline(z_trans, color='gray', ls=':', lw=1, alpha=0.5,
           label=f'z_trans = {z_trans:.3f}')
ax.set_xlabel('Redshift z')
ax.set_ylabel('CDM mass (normalized)')
ax.set_title('Null test: Mass Evolution (CDM Drain)')
ax.legend(fontsize=9)
ax.invert_xaxis()

# Panel C: Residual (sim - analytical) / analytical
ax = axes[2]
residual = (mass_null_norm - fcdm_at_snaps) / fcdm_at_snaps * 100
ax.plot(z_null, residual, 'go-', ms=6, lw=2)
ax.axhline(0, color='gray', ls=':', lw=1)
ax.axhline(0.05, color='red', ls='--', lw=1, alpha=0.5, label='±0.05%')
ax.axhline(-0.05, color='red', ls='--', lw=1, alpha=0.5)
ax.set_xlabel('Redshift z')
ax.set_ylabel('Residual (%)')
ax.set_title('Null test: Drain Residual (sim − analytical)')
ax.legend(fontsize=9)
ax.invert_xaxis()

plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/fig_NB07_null_test_test.pdf', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] §2c Null test visualization complete')

---

## §3. EU Production: P(k) Evolution

Production was run with full **TreePM** gravity. This is the production simulation:
CDM drain + gravitational clustering.

We expect:
- Structure growth from z=49 (near-flat ICs) to z=0 (full nonlinear P(k))
- ~10⁴× growth in P(k) amplitude at intermediate k
- Nonlinear turnover at high k (k > 1 h/Mpc)
- Drain-induced suppression relative to ΛCDM (quantified in §6)

In [ ]:
# §3a. Production P(k) EVOLUTION — All 15 snapshots

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Panel A: Full P(k) evolution ──
ax = axes[0]
cmap = plt.cm.plasma
z_vals = [s['z'] for s in prod_snaps]

for i, snap in enumerate(prod_snaps):
    color = cmap(1.0 - i / 14.0)  # z=49 → dark, z=0 → bright
    mask = snap['nmodes'] > 10
    label = f'z={snap["z"]:.1f}' if i in [0, 1, 4, 8, 11, 14] else ''
    ax.loglog(snap['k_hmpc'][mask], snap['Pk_mpch3'][mask],
              color=color, alpha=0.85, lw=1.2, label=label)

# k_Nyquist
k_nyq = np.pi * 1024 / BOX_MPC_H  # π * N / L
ax.axvline(k_nyq, color='red', ls='--', lw=1, alpha=0.5,
           label=f'k_Nyq = {k_nyq:.1f}')

ax.set_xlabel('k [h/Mpc]')
ax.set_ylabel('P(k) [(Mpc/h)³]')
ax.set_title('EU Production: P(k) Evolution z=49 → z=0')
ax.legend(fontsize=8, ncol=2, loc='lower left')
ax.set_xlim(0.01, 6)

# ── Panel B: Growth at k_ref ──
ax = axes[1]
k_ref = 0.1  # h/Mpc — well in linear regime

z_arr_v3 = np.array([s['z'] for s in prod_snaps])
pk_at_kref = []
for snap in prod_snaps:
    # Interpolate P(k) at k_ref
    idx = np.argmin(np.abs(snap['k_hmpc'] - k_ref))
    pk_at_kref.append(snap['Pk_mpch3'][idx])
pk_at_kref = np.array(pk_at_kref)

ax.semilogy(z_arr_v3, pk_at_kref, 'ro-', ms=7, lw=2,
            label=f'P(k={k_ref}) EU')
ax.axvline(z_trans, color='gray', ls=':', lw=1, alpha=0.5,
           label=f'z_trans = {z_trans:.3f}')
ax.set_xlabel('Redshift z')
ax.set_ylabel('P(k) [(Mpc/h)³]')
ax.set_title(f'EU Prod: P(k = {k_ref} h/Mpc) vs Redshift')
ax.legend(fontsize=9)
ax.invert_xaxis()

plt.tight_layout()
plt.savefig('figures/fig_NB07_prod_pk_evolution.pdf', dpi=150,
            bbox_inches='tight')
plt.show()

# Growth factor
growth_total = pk_at_kref[-1] / pk_at_kref[0]
print(f'\nP(k={k_ref}) growth from z={z_arr_v3[0]:.0f} to z=0: '
      f'{growth_total:.1f}×')
print(f'  z=49: P = {pk_at_kref[0]:.4e}')
print(f'  z=0:  P = {pk_at_kref[-1]:.4e}')
print('[OK] §3a Production P(k) evolution visualization complete')

In [ ]:
# §3b. Production P(k) SHAPE ANALYSIS — z=0, z≈1, z≈2

snap_z0 = prod_snaps[-1]  # z = 0
snap_z1 = prod_snaps[12]  # z ≈ 1
snap_z2 = prod_snaps[10]  # z ≈ 2

fig, ax = plt.subplots(figsize=(12, 6))

for snap, color, ls in [(snap_z0, 'navy', '-'),
                         (snap_z1, 'crimson', '--'),
                         (snap_z2, 'forestgreen', ':')]:
    mask = snap['nmodes'] > 20
    ax.loglog(snap['k_hmpc'][mask], snap['Pk_mpch3'][mask],
              color=color, lw=2, ls=ls,
              label=f'z = {snap["z"]:.2f}')

ax.axvline(k_nyq, color='red', ls='--', lw=1, alpha=0.4,
           label=f'k_Nyq = {k_nyq:.1f}')
ax.axvline(0.1, color='gray', ls=':', lw=1, alpha=0.4,
           label='k = 0.1 (linear)')
ax.axvline(0.5, color='gray', ls=':', lw=1, alpha=0.3,
           label='k = 0.5 (quasi-linear)')

ax.set_xlabel('k [h/Mpc]', fontsize=14)
ax.set_ylabel('P(k) [(Mpc/h)³]', fontsize=14)
ax.set_title('EU Production: Matter Power Spectrum', fontsize=14)
ax.legend(fontsize=10)
ax.set_xlim(0.01, 6)

plt.tight_layout()
plt.savefig('figures/fig_NB07_prod_pk_z0.pdf', dpi=150,
            bbox_inches='tight')
plt.show()

# Report key values
for snap in [snap_z0, snap_z1, snap_z2]:
    idx01 = np.argmin(np.abs(snap['k_hmpc'] - 0.1))
    idx1 = np.argmin(np.abs(snap['k_hmpc'] - 1.0))
    print(f'z={snap["z"]:5.2f}  '
          f'P(k=0.1)={snap["Pk_mpch3"][idx01]:.2e}  '
          f'P(k=1.0)={snap["Pk_mpch3"][idx1]:.2e}  '
          f'mass={snap["mass"]:.10f}')

print('\n[OK] §3b Production shape analysis complete')

---

## §4. Heaviside Cutoff Validation

The EU drain activates at $z_{\mathrm{trans}} = 5.986$ via a smoothed Heaviside
(logistic) function. Above $z_{\mathrm{trans}}$, the drain rate is **exactly zero**.

### Test:
- Snapshots 000 (z=49), 001 (z≈7.0), 002 (z≈6.0) are **above** z_trans → mass must be identical
- Snapshot 003 (z≈5.5) is the **first below** z_trans → mass must drop
- The Heaviside step must be sharp: $|m(z=7) - m(z=49)| / m(z=49) \approx 0$

In [ ]:
# §4a. HEAVISIDE CUTOFF — PRODUCTION

z_prod = np.array([s['z'] for s in prod_snaps])
mass_prod = np.array([s['mass'] for s in prod_snaps])

print('=== HEAVISIDE CUTOFF VALIDATION (Production) ===')
print(f'z_trans (NB01) = {z_trans:.4f}\n')
print(f'{"Snap":>4} {"z":>8} {"mass":>14} '
      f'{"Above z_t?":>10} {"\u0394m from snap000":>18}')
print('-' * 60)

mass_ref = mass_prod[0]  # z=49, reference
heaviside_results = []

for i, snap in enumerate(prod_snaps):
    above = snap['z'] >= z_trans
    dm = abs(snap['mass'] - mass_ref) / mass_ref * 100
    status = 'YES' if above else 'no'
    heaviside_results.append({
        'snap_id': snap['snap_id'], 'z': snap['z'],
        'mass': snap['mass'], 'above_zt': above,
        'delta_pct': dm
    })
    marker = '  \u25c4\u2500\u2500 TRANSITION' if i == 3 else ''
    print(f'{snap["snap_id"]:4d} {snap["z"]:8.2f} '
          f'{snap["mass"]:14.10f} {status:>10} '
          f'{dm:18.6f}%{marker}')

# Validate: first 3 snapshots (z > z_trans) same mass
m_above = mass_prod[:3]
max_spread = ((np.max(m_above) - np.min(m_above))
              / np.mean(m_above) * 100)
print(f'\nMass spread above z_trans (snap 000-002): '
      f'{max_spread:.6f}%')

# Validate: first drop at snap 003
first_drop = ((mass_prod[2] - mass_prod[3])
              / mass_prod[2] * 100)
print(f'First drop (snap 002 \u2192 003, z\u22486.0 \u2192 z\u22485.5): '
      f'{first_drop:.4f}%')

assert max_spread < 0.01, \
    f'Mass varies above z_trans: spread = {max_spread:.6f}%'
assert first_drop > 0.01, \
    f'No mass drop at z < z_trans: drop = {first_drop:.6f}%'

print('\n[OK] Heaviside validated:')
print(f'  - Mass constant above z_trans (spread < 0.01%): \u2705')
print(f'  - First drop below z_trans: {first_drop:.4f}% \u2705')

In [ ]:
# §4b. HEAVISIDE VISUALIZATION

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Panel A: Mass vs redshift with z_trans marker ──
ax = axes[0]
ax.plot(z_prod, mass_prod, 'bo-', ms=8, lw=2,
        label='EU (prod) mass', zorder=5)
ax.axvline(z_trans, color='red', ls='-', lw=2, alpha=0.7,
           label=f'z_trans = {z_trans:.3f}')
ax.axhspan(mass_prod[0] - 0.0001, mass_prod[0] + 0.0001,
           color='green', alpha=0.15, label='IC mass \u00b1 0.01%')

# Annotate key points
ax.annotate(
    f'z={prod_snaps[2]["z"]:.2f}\n(above z_t)',
    xy=(prod_snaps[2]['z'], prod_snaps[2]['mass']),
    xytext=(prod_snaps[2]['z'] + 1,
            prod_snaps[2]['mass'] + 0.002),
    fontsize=8,
    arrowprops=dict(arrowstyle='->', lw=0.8))
ax.annotate(
    f'z={prod_snaps[3]["z"]:.2f}\n(FIRST DROP)',
    xy=(prod_snaps[3]['z'], prod_snaps[3]['mass']),
    xytext=(prod_snaps[3]['z'] - 1.5,
            prod_snaps[3]['mass'] + 0.005),
    fontsize=8, color='red',
    arrowprops=dict(arrowstyle='->', color='red', lw=0.8))

ax.set_xlabel('Redshift z')
ax.set_ylabel('CDM mass (internal units)')
ax.set_title('Heaviside Cutoff: Mass vs Redshift')
ax.legend(fontsize=8)
ax.set_xlim(0, 12)
ax.invert_xaxis()

# ── Panel B: Null test vs Production normalized ──
ax = axes[1]
mass_null_arr = np.array([s['mass'] for s in null_snaps])

# Normalize both to their z=49 values
m3_norm = mass_prod / mass_prod[0]
m2_norm = mass_null_arr / mass_null_arr[0]

ax.plot(z_prod, m3_norm, 'bs-', ms=7, lw=2,
        label='EU (prod) (gravity ON)')
ax.plot(z_null, m2_norm, 'r^--', ms=7, lw=2,
        label='Null test (gravity OFF)', alpha=0.8)
ax.axvline(z_trans, color='gray', ls=':', lw=1.5, alpha=0.7,
           label=f'z_trans = {z_trans:.3f}')

ax.set_xlabel('Redshift z')
ax.set_ylabel('mass / mass(z=49)')
ax.set_title('Null test vs Production: Identical Drain (Heaviside Proof)')
ax.legend(fontsize=9)
ax.invert_xaxis()

plt.tight_layout()
plt.savefig('figures/fig_NB07_heaviside.pdf', dpi=150,
            bbox_inches='tight')
plt.show()

# Final: max Null-Prod drain difference
drain_diff = np.max(np.abs(m3_norm - m2_norm))
print(f'\nMax |V3_norm - V2_norm| across all snapshots: '
      f'{drain_diff:.2e}')
print('[OK] §4b Heaviside visualization complete')

---

## §5. Drain Validation: Null test vs Production vs Analytical

The CDM drain is implemented at the **background level** via `eu_tables.h`.
It rescales particle masses at each timestep according to $f_{\mathrm{cdm}}(a)$.

### Critical test:
The drain must be **identical** in Null test (no gravity) and Production (full gravity),
because the drain is a background-level mass rescaling that does not
couple to gravitational dynamics. Both must match the NB01 analytical
prediction $f_{\mathrm{cdm}}(z)$.

### Metrics:
- `max |mass_null - mass_prod| / mass_null` → should be < 0.01%
- `|drain_total - drain_NB01| / drain_NB01` → should be < 0.1%
- Total drain at z=0: ~4.42% of initial CDM mass

In [ ]:
# §5a. DRAIN COMPARISON: Null test vs Production vs ANALYTICAL

z_prod = np.array([s['z'] for s in prod_snaps])
mass_prod = np.array([s['mass'] for s in prod_snaps])
z_null = np.array([s['z'] for s in null_snaps])
mass_null = np.array([s['mass'] for s in null_snaps])

# Analytical f_cdm at snapshot redshifts
fcdm_prod = compute_fcdm(z_prod, z_trans, b_exp)
fcdm_at_z49 = compute_fcdm(np.array([49.0]),
                           z_trans, b_exp)[0]

# Normalize simulation masses to analytical convention
mass_prod_norm = mass_prod / mass_prod[0] * fcdm_at_z49
mass_null_norm = mass_null / mass_null[0] * fcdm_at_z49

print('=== DRAIN COMPARISON: Null test vs Production vs ANALYTICAL ===')
print(f'{"Snap":>4} {"z":>7} {"V2_norm":>11} '
      f'{"V3_norm":>11} {"Analytical":>11} '
      f'{"Null-Prod(%)":>9} {"Prod-Ana(%)":>10}')
print('-' * 72)

drain_comparison = []
for i in range(len(prod_snaps)):
    dv2v3 = abs(mass_null_norm[i] - mass_prod_norm[i]) / mass_null_norm[i] * 100
    dv3ana = (mass_prod_norm[i] - fcdm_prod[i]) / fcdm_prod[i] * 100
    drain_comparison.append({
        'snap_id': prod_snaps[i]['snap_id'],
        'z': prod_snaps[i]['z'],
        'mass_null_norm': float(mass_null_norm[i]),
        'mass_prod_norm': float(mass_prod_norm[i]),
        'fcdm_analytical': float(fcdm_prod[i]),
        'delta_v2v3_pct': float(dv2v3),
        'delta_v3ana_pct': float(dv3ana)
    })
    print(f'{prod_snaps[i]["snap_id"]:4d} {prod_snaps[i]["z"]:7.2f} '
          f'{mass_null_norm[i]:11.8f} {mass_prod_norm[i]:11.8f} '
          f'{fcdm_prod[i]:11.8f} {dv2v3:9.5f} {dv3ana:+10.5f}')

max_v2v3 = max(d['delta_v2v3_pct'] for d in drain_comparison)
max_v3ana = max(abs(d['delta_v3ana_pct']) for d in drain_comparison)

print(f'\nMax |Null - Prod| / Null:       {max_v2v3:.6f}%')
print(f'Max |Prod - Ana| / Ana:     {max_v3ana:.6f}%')

assert max_v2v3 < 0.1, f'Null/Prod drain mismatch: {max_v2v3:.6f}%'
print('\n[OK] Null test and Production drain are consistent')

In [ ]:
# §5b. TOTAL DRAIN QUANTIFICATION

# Drain = 1 - mass(z=0) / mass(z_IC)
# Note: z_IC = 49, which is above z_trans,
# so drain between z=\infty and z=49 is negligible

drain_null = (1.0 - mass_null[-1] / mass_null[0]) * 100
drain_prod = (1.0 - mass_prod[-1] / mass_prod[0]) * 100
drain_ana = (1.0 - fcdm_z0_exact / fcdm_at_z49) * 100

print('=== TOTAL CDM DRAIN ===')
print(f'  Null test (no gravity):  {drain_null:.4f}%')
print(f'  EU (production):  {drain_prod:.4f}%')
print(f'  NB01 analytical:  {drain_ana:.4f}%')
print(f'  |Null - Prod|:        {abs(drain_null - drain_prod):.4f}%')
print(f'  |Prod - Ana|:       {abs(drain_prod - drain_ana):.4f}%')

# Cross-check with NB01 f_cdm(z=0)
print(f'\n  NB01 f_cdm(z=0) = {fcdm_z0_exact}')
print(f'  Production mass(z=0)/mass(z=49) = '
      f'{mass_prod[-1]/mass_prod[0]:.10f}')
print(f'  Production mass_norm(z=0) = {mass_prod_norm[-1]:.10f}')

# Validate the headline number: ~4.42%
assert abs(drain_prod - 4.42) < 0.5, \
    f'Drain outside expected range: {drain_prod:.4f}%'
assert abs(drain_null - drain_prod) < 0.01, \
    f'Null/Prod drain mismatch: {abs(drain_null - drain_prod):.4f}%'

print('\n[OK] §5b Total drain validated: ~4.42% \u2705')

In [ ]:
# §5c. DRAIN VISUALIZATION — Triple comparison

z_fine = np.linspace(0, 49, 500)
fcdm_fine = compute_fcdm(z_fine, z_trans, b_exp)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Panel A: Mass curves (all three) ──
ax = axes[0]
ax.plot(z_prod, mass_prod_norm, 'bs-', ms=7, lw=2,
        label='EU (prod) (gravity ON)', zorder=5)
ax.plot(z_null, mass_null_norm, 'r^--', ms=7, lw=1.5,
        label='Null test (gravity OFF)', alpha=0.8)
ax.plot(z_fine, fcdm_fine, 'k-', lw=1.5,
        label='NB01 analytical', alpha=0.6)
ax.axvline(z_trans, color='gray', ls=':', lw=1, alpha=0.5,
           label=f'z_trans = {z_trans:.3f}')

ax.set_xlabel('Redshift z', fontsize=12)
ax.set_ylabel('$f_{\\mathrm{cdm}}(z)$', fontsize=12)
ax.set_title('CDM Mass Fraction: Simulation vs Theory')
ax.legend(fontsize=9)
ax.invert_xaxis()

# ── Panel B: Null test vs Production residual ──
ax = axes[1]
resid_v2v3 = ((mass_prod_norm - mass_null_norm)
              / mass_null_norm * 100)
ax.plot(z_prod, resid_v2v3, 'go-', ms=6, lw=2)
ax.axhline(0, color='gray', ls=':', lw=1)
ax.fill_between([0, 50], -0.01, 0.01,
                color='green', alpha=0.1,
                label='\u00b10.01% band')
ax.set_xlabel('Redshift z', fontsize=12)
ax.set_ylabel('(Prod - Null) / Null  [%]', fontsize=12)
ax.set_title('Drain Consistency: Production vs Null test')
ax.legend(fontsize=9)
ax.invert_xaxis()
ax.set_xlim(0, 50)

# ── Panel C: Prod vs Analytical residual ──
ax = axes[2]
resid_v3ana = ((mass_prod_norm - fcdm_prod)
               / fcdm_prod * 100)
ax.plot(z_prod, resid_v3ana, 'mo-', ms=6, lw=2)
ax.axhline(0, color='gray', ls=':', lw=1)
ax.fill_between([0, 50], -0.05, 0.05,
                color='purple', alpha=0.1,
                label='\u00b10.05% band')
ax.set_xlabel('Redshift z', fontsize=12)
ax.set_ylabel('(Prod - Analytical) / Analytical  [%]',
              fontsize=12)
ax.set_title('Drain Accuracy: Prod vs NB01')
ax.legend(fontsize=9)
ax.invert_xaxis()
ax.set_xlim(0, 50)

plt.tight_layout()
plt.savefig('figures/fig_NB07_drain_validation.pdf',
            dpi=150, bbox_inches='tight')
plt.show()

print('[OK] §5c Drain validation visualization complete')
print(f'\n=== PHASE 1 COMPLETE ===')
print(f'  \u2705 \u00a71 Data loaded (30 P(k) files)')
print(f'  \u2705 \u00a72 Null test passed')
print(f'  \u2705 \u00a73 Production P(k) evolution validated')
print(f'  \u2705 \u00a74 Heaviside cutoff confirmed')
print(f'  \u2705 \u00a75 Drain Null=Prod=Analytical ({drain_prod:.2f}%)')

---

## §6. Suppression Ratio: $P_{\mathrm{EU}}(k) \, / \, P_{\Lambda\mathrm{CDM}}(k)$

This is the **key physics plot** for Paper I. It shows how the CDM drain
modifies the nonlinear matter power spectrum relative to standard ΛCDM.

We compute ΛCDM $P(k)$ using **CLASS** with identical cosmological
parameters (Planck 2018 + C2 posteriors), then take the ratio:

$$S(k, z) = \frac{P_{\mathrm{EU}}^{\mathrm{N\text{-}body}}(k, z)}{P_{\Lambda\mathrm{CDM}}^{\mathrm{CLASS}}(k, z)}$$

### Expected behavior:
- **Large scales** ($k < 0.1$): $S \approx (\sigma_{8,\mathrm{EU}} / \sigma_{8,\Lambda})^2 \sim 0.98$–$0.99$ (linear growth suppression)
- **Small scales** ($k > 1$): enhanced suppression (anemic halos from CDM loss)
- **Transition**: smooth crossover around $k \sim 0.3$–$0.5$ h/Mpc

In [ ]:
# §6a. COMPUTE ΛCDM P(k) VIA CLASS

# Install CLASS if needed (Colab)
try:
    from classy import Class
    print('[OK] classy already installed')
except ImportError:
    import subprocess, sys
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', 'cython'])
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', 'classy'])
    from classy import Class
    print('[OK] classy installed successfully')

# ΛCDM parameters — identical to EU MCMC C2
# (same background, but NO drain)
# Must include neutrinos for consistent sigma8
lcdm_params = {
    'h': h_EU,
    'omega_b': omega_b,
    'omega_cdm': omega_cdm_prim,
    'n_s': 0.9649,
    'ln10^{10}A_s': 3.044,
    'tau_reio': 0.0544,
    'N_ur': 2.0328,
    'N_ncdm': 1,
    'm_ncdm': 0.0589,
    'output': 'mPk',
    'P_k_max_h/Mpc': 10.0,
    'z_max_pk': 50.0,
    'non_linear': 'hmcode',
}

cosmo_lcdm = Class()
cosmo_lcdm.set(lcdm_params)
cosmo_lcdm.compute()

print('[OK] ΛCDM CLASS computed')
print(f'  H0 = {cosmo_lcdm.h() * 100:.3f} km/s/Mpc')
print(f'  sigma8 = {cosmo_lcdm.sigma8():.6f}')
print(f'  Omega_m = '
      f'{cosmo_lcdm.Omega_m():.6f}')


In [ ]:
# §6b. SUPPRESSION RATIO AT z=0

snap_z0 = prod_snaps[-1]  # z = 0

# N-body P(k) at z=0
k_eu = snap_z0['k_hmpc']       # h/Mpc
pk_eu = snap_z0['Pk_mpch3']    # (Mpc/h)^3
nmodes = snap_z0['nmodes']

# Cut low-Nmodes bins (shot noise)
mask = nmodes > 20
k_eu = k_eu[mask]
pk_eu = pk_eu[mask]

# CLASS P(k) at same k values
# CLASS uses k in 1/Mpc, P(k) in Mpc^3
# Convert: k_mpc = k_hmpc * h, Pk_mpch3 = Pk_mpc3 * h^3
pk_lcdm = np.array([
    cosmo_lcdm.pk(ki * h_EU, 0.0) * h_EU**3
    for ki in k_eu
])  # Now in (Mpc/h)^3

# Suppression ratio
S_k = pk_eu / pk_lcdm

print('=== SUPPRESSION RATIO P_EU / P_LCDM at z=0 ===')
print(f'{"k [h/Mpc]":>12} {"P_EU":>14} '
      f'{"P_LCDM":>14} {"S(k)":>10}')
print('-' * 55)
# Print at selected k values
for ki_target in [0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 3.0]:
    idx = np.argmin(np.abs(k_eu - ki_target))
    print(f'{k_eu[idx]:12.4f} {pk_eu[idx]:14.4e} '
          f'{pk_lcdm[idx]:14.4e} {S_k[idx]:10.4f}')

# Report summary
mask_lin = k_eu < 0.1
mask_nl = k_eu > 1.0
S_linear = np.mean(S_k[mask_lin])
S_nonlin = np.mean(S_k[mask_nl & (k_eu < 3.0)])
print(f'\nMean S(k<0.1)  = {S_linear:.4f} '
      f'(linear regime)')
print(f'Mean S(1<k<3)  = {S_nonlin:.4f} '
      f'(nonlinear regime)')
print(f'\n[OK] §6b Suppression ratio computed')

In [ ]:
# §6c. SUPPRESSION RATIO AT MULTIPLE REDSHIFTS

target_snaps = [
    (14, 'z=0.00'),
    (13, 'z=0.50'),
    (12, 'z=1.00'),
    (10, 'z=2.00'),
]

suppression_data = {}

for snap_idx, label in target_snaps:
    snap = prod_snaps[snap_idx]
    z = snap['z']
    k_s = snap['k_hmpc']
    pk_s = snap['Pk_mpch3']
    nm = snap['nmodes']

    mask = nm > 20
    k_s = k_s[mask]
    pk_s = pk_s[mask]

    # CLASS P(k,z)
    pk_lcdm_z = np.array([
        cosmo_lcdm.pk(ki * h_EU, z) * h_EU**3
        for ki in k_s
    ])

    S = pk_s / pk_lcdm_z
    suppression_data[label] = {
        'k': k_s, 'S': S, 'z': z,
        'pk_eu': pk_s, 'pk_lcdm': pk_lcdm_z
    }
    print(f'{label}: S(k=0.1)={S[np.argmin(np.abs(k_s-0.1))]:.4f}  '
          f'S(k=1.0)={S[np.argmin(np.abs(k_s-1.0))]:.4f}')

print('[OK] §6c Multi-z suppression computed')

In [ ]:
# §6d. SUPPRESSION RATIO VISUALIZATION

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# -- Panel A: Multi-z S(k) --
ax = axes[0]
colors = ['navy', 'crimson', 'forestgreen', 'darkorange']
for i, (label, data) in enumerate(suppression_data.items()):
    # Smooth with running median for clarity
    from scipy.ndimage import uniform_filter1d
    S_smooth = uniform_filter1d(data['S'], size=5)
    ax.semilogx(data['k'], S_smooth,
                color=colors[i], lw=2, label=label)

ax.axhline(1.0, color='gray', ls=':', lw=1, alpha=0.5)
ax.axhline(0.98, color='gray', ls='--', lw=0.8,
           alpha=0.3)

# CUB line (flat suppression from sigma8 ratio)
sigma8_eu = 0.8274    # MCMC C2
sigma8_lcdm_val = cosmo_lcdm.sigma8()
S_CUB = (sigma8_eu / sigma8_lcdm_val)**2
ax.axhline(S_CUB, color='purple', ls='--', lw=1.5,
           alpha=0.6,
           label=f'CUB = $(\\sigma_8^{{EU}} / \\sigma_8^{{\\Lambda}})^2$'
                 f' = {S_CUB:.4f}')

ax.set_xlabel('k [h/Mpc]', fontsize=13)
ax.set_ylabel(
    '$S(k) = P_{\\mathrm{EU}}(k) \\, / \\, '
    'P_{\\Lambda\\mathrm{CDM}}(k)$',
    fontsize=13)
ax.set_title('Suppression Ratio: EU N-body vs '
             '\u039bCDM (CLASS+HMCode)', fontsize=13)
ax.legend(fontsize=9, loc='lower left')
ax.set_xlim(0.02, 5)
ax.set_ylim(0.7, 1.15)

# -- Panel B: P(k) comparison at z=0 --
ax = axes[1]
d0 = suppression_data['z=0.00']
ax.loglog(d0['k'], d0['pk_eu'],
          'navy', lw=2, label='EU (N-body EU)')
ax.loglog(d0['k'], d0['pk_lcdm'],
          'red', lw=2, ls='--',
          label='\u039bCDM (CLASS+HMCode)')

ax.set_xlabel('k [h/Mpc]', fontsize=13)
ax.set_ylabel('P(k) [(Mpc/h)\u00b3]', fontsize=13)
ax.set_title('Matter Power Spectrum at z=0',
             fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0.02, 5)

plt.tight_layout()
plt.savefig('figures/fig_NB07_suppression_ratio.pdf',
            dpi=150, bbox_inches='tight')
plt.show()

# Cleanup CLASS
cosmo_lcdm.struct_cleanup()
cosmo_lcdm.empty()

print('[OK] §6d Suppression ratio visualization complete')

---

## §7. Null test vs Production: The Definitive Comparison

This section presents the **strongest argument** for a referee:

- **Null test** (no gravity): particles don't cluster → P(k) is flat white noise
- **Production** (full gravity): particles cluster normally → full nonlinear P(k)
- **Both** have **identical CDM drain** → proves drain is purely thermodynamic

The mass evolution is identical in both runs because the drain is implemented
as a background-level particle mass rescaling in `run.cc`, completely
decoupled from gravitational force computation.

### Key insight:
If the drain had *any* coupling to gravity (e.g., through force softening
or tree-walk), the mass curves would diverge. They don't.

In [ ]:
# §7a. Null test vs Production — P(k) COMPARISON

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# -- Panel A: P(k) at z=0 --
ax = axes[0, 0]
s3 = prod_snaps[-1]  # z=0 Production
s2 = null_snaps[-1]  # z=0 Null test
mask3 = s3['nmodes'] > 10
mask2 = s2['nmodes'] > 10

ax.loglog(s3['k_hmpc'][mask3], s3['Pk_mpch3'][mask3],
          'navy', lw=2, label='EU (prod) (gravity ON)')
ax.loglog(s2['k_hmpc'][mask2], s2['Pk_mpch3'][mask2],
          'red', lw=2, ls='--', alpha=0.8,
          label='Null test (gravity OFF)')

ax.set_xlabel('k [h/Mpc]')
ax.set_ylabel('P(k) [(Mpc/h)\u00b3]')
ax.set_title('z = 0: P(k) Comparison')
ax.legend(fontsize=10)
ax.set_xlim(0.01, 6)

# -- Panel B: P(k) ratio Prod/Null at z=0 --
ax = axes[0, 1]
# Interpolate Null onto Prod k-grid
from scipy.interpolate import interp1d
f_v2 = interp1d(s2['k_hmpc'], s2['Pk_mpch3'],
                kind='linear', fill_value='extrapolate')
pk_v2_interp = f_v2(s3['k_hmpc'][mask3])
ratio_v3v2 = s3['Pk_mpch3'][mask3] / pk_v2_interp

ax.semilogx(s3['k_hmpc'][mask3], ratio_v3v2,
            'forestgreen', lw=2)
ax.axhline(1.0, color='gray', ls=':', lw=1)
ax.set_xlabel('k [h/Mpc]')
ax.set_ylabel('P_V3(k) / P_V2(k)')
ax.set_title('z = 0: Gravity effect (Prod/Null ratio)')
ax.set_xlim(0.01, 6)

# Annotate
ax.annotate(
    f'At k=0.1: ratio = '
    f'{ratio_v3v2[np.argmin(np.abs(s3["k_hmpc"][mask3]-0.1))]:.0f}\u00d7',
    xy=(0.1, ratio_v3v2[np.argmin(np.abs(s3['k_hmpc'][mask3]-0.1))]),
    fontsize=9, color='forestgreen')

# -- Panel C: Mass evolution overlay --
ax = axes[1, 0]
z_prod = np.array([s['z'] for s in prod_snaps])
z_null = np.array([s['z'] for s in null_snaps])
mass_prod = np.array([s['mass'] for s in prod_snaps])
mass_null = np.array([s['mass'] for s in null_snaps])

ax.plot(z_prod, mass_prod, 'bs-', ms=8, lw=2,
        label='EU (prod) (gravity ON)', zorder=5)
ax.plot(z_null, mass_null, 'r^--', ms=8, lw=2,
        label='Null test (gravity OFF)', alpha=0.8)
ax.axvline(z_trans, color='gray', ls=':', lw=1.5,
           alpha=0.5,
           label=f'z_trans = {z_trans:.3f}')

ax.set_xlabel('Redshift z')
ax.set_ylabel('CDM mass (internal units)')
ax.set_title('Mass Evolution: Identical Drain')
ax.legend(fontsize=9)
ax.invert_xaxis()

# -- Panel D: Mass residual Prod - Null --
ax = axes[1, 1]
mass_resid = (mass_prod - mass_null) / mass_null * 100
ax.plot(z_prod, mass_resid, 'go-', ms=7, lw=2)
ax.axhline(0, color='gray', ls=':', lw=1)
ax.fill_between([0, 50], -0.01, 0.01,
                color='green', alpha=0.1,
                label='\u00b10.01% band')

ax.set_xlabel('Redshift z')
ax.set_ylabel('(mass_prod - mass_null) / mass_null  [%]')
ax.set_title('Drain Residual: Production vs Null test')
ax.legend(fontsize=9)
ax.invert_xaxis()
ax.set_xlim(0, 50)

plt.tight_layout()
plt.savefig('figures/fig_NB07_null_vs_prod.pdf',
            dpi=150, bbox_inches='tight')
plt.show()

# Summary statistics
max_mass_resid = np.max(np.abs(mass_resid))
print(f'\n=== Null test vs Production SUMMARY ===')
print(f'  Max |mass residual|: '
      f'{max_mass_resid:.6f}%')
print(f'  P(k) ratio at z=0, k=0.1: '
      f'{ratio_v3v2[np.argmin(np.abs(s3["k_hmpc"][mask3]-0.1))]:.0f}\u00d7')
print(f'  Conclusion: drain is purely '
      f'thermodynamic (decoupled from gravity)')
print('[OK] \u00a77a Null test vs Production comparison complete')

In [ ]:
# \u00a77b. Null test vs Production \u2014 DIAGNOSTIC SUMMARY TABLE

print('=' * 65)
print('  Null test vs Production DIAGNOSTIC SUMMARY')
print('=' * 65)
print(f'{"Metric":.<40} {"Null":>10} {"Prod":>10}')
print('-' * 65)
print(f'{"SELFGRAVITY":.<40} {"OFF":>10} {"ON":>10}')
print(f'{"Particles":.<40} '
      f'{NPART:>10,} {NPART:>10,}')
print(f'{"Box [Mpc/h]":.<40} '
      f'{BOX_MPC_H:>10.0f} {BOX_MPC_H:>10.0f}')
print(f'{"mass(z=49)":.<40} '
      f'{mass_null[0]:>10.6f} {mass_prod[0]:>10.6f}')
print(f'{"mass(z=0)":.<40} '
      f'{mass_null[-1]:>10.6f} {mass_prod[-1]:>10.6f}')
print(f'{"Total drain [%]":.<40} '
      f'{(1-mass_null[-1]/mass_null[0])*100:>10.4f} '
      f'{(1-mass_prod[-1]/mass_prod[0])*100:>10.4f}')
print(f'{"P(k=0.1, z=0) [(Mpc/h)^3]":.<40} '
      f'{null_snaps[-1]["Pk_mpch3"][np.argmin(np.abs(null_snaps[-1]["k_hmpc"]-0.1))]:>10.2e} '
      f'{prod_snaps[-1]["Pk_mpch3"][np.argmin(np.abs(prod_snaps[-1]["k_hmpc"]-0.1))]:>10.2e}')
print(f'{"Structure formed?":.<40} '
      f'{"NO":>10} {"YES":>10}')
print(f'{"Max |drain residual| [%]":.<40} '
      f'{"-":>10} {max_mass_resid:>10.6f}')
print('=' * 65)
print('\nConclusion: CDM drain is background-level '
      '(identical with/without gravity).')
print('This proves the drain does not couple '
      'to gravitational dynamics.')
print('[OK] \u00a77b Diagnostic summary complete')

---

## §8. Growth Factor $D(z)$ Extraction

Extract the linear growth factor $D(z)$ from the N-body P(k) at a reference
scale in the linear regime ($k_{\mathrm{ref}} \approx 0.05$ h/Mpc), and compare
with the analytical EU prediction from the growth ODE.

$$D(z) = \sqrt{\frac{P(k_{\mathrm{ref}}, z)}{P(k_{\mathrm{ref}}, z=0)}}$$

### Growth suppression:
The EU model predicts slightly suppressed growth relative to ΛCDM due to
CDM mass loss. The growth suppression factor $g = D_{\mathrm{EU}}(0) / D_{\Lambda}(0)$
should match the NB03/NB09 analytical value ($g \approx 0.993$).

In [ ]:
# §8a. EXTRACT D(z) FROM N-BODY P(k)

k_ref = 0.05  # h/Mpc — safely in linear regime

# Production D(z)
z_prod = np.array([s['z'] for s in prod_snaps])
pk_kref_v3 = []
for snap in prod_snaps:
    idx = np.argmin(np.abs(snap['k_hmpc'] - k_ref))
    pk_kref_v3.append(snap['Pk_mpch3'][idx])
pk_kref_v3 = np.array(pk_kref_v3)

# D(z) = sqrt(P(k_ref, z) / P(k_ref, z=0))
D_v3 = np.sqrt(pk_kref_v3 / pk_kref_v3[-1])  # [-1] = z=0

print('=== GROWTH FACTOR D(z) FROM N-BODY ===')
print(f'Reference scale: k_ref = {k_ref} h/Mpc')
print(f'P(k_ref, z=0) = {pk_kref_v3[-1]:.4e}\n')
print(f'{"Snap":>4} {"z":>7} {"P(k_ref)":>12} '
      f'{"D(z)":>8} {"D/D(0)":>8}')
print('-' * 45)
for i, snap in enumerate(prod_snaps):
    print(f'{snap["snap_id"]:4d} {snap["z"]:7.2f} '
          f'{pk_kref_v3[i]:12.4e} {D_v3[i]:8.5f} '
          f'{D_v3[i]/D_v3[-1]:8.5f}')

print(f'\nD(z=0) = {D_v3[-1]:.6f} (should be 1.0 by definition)')
print('[OK] §8a Growth factor extracted')

In [ ]:
# §8b. ANALYTICAL GROWTH FACTOR — EU vs ΛCDM

from scipy.integrate import solve_ivp

# Pre-compute fc0 for normalization
fc0 = compute_fcdm(np.array([0.0]), z_trans, b_exp)[0]

# Neutrino physical density (from m_nu = 0.0589 eV used in CLASS §6a)
omega_nu = 0.000632

def growth_ode_eu(lna, y, z_t, b_val):
    """Growth ODE for EU model.
    Component-level: CDM (drained), baryons, neutrinos.
    y = [D, dD/dlna]"""
    a = np.exp(lna)
    z = 1.0 / a - 1.0

    # f_cdm(z) applies drain ONLY to CDM
    fc = compute_fcdm(np.array([z]), z_t, b_val)[0]

    # Physical densities at scale factor a
    Om_cdm_a = (omega_cdm_prim / h_EU**2) * fc * a**(-3)
    Om_b_a   = (omega_b / h_EU**2) * a**(-3)
    Om_nu_a  = (omega_nu / h_EU**2) * a**(-3)

    # Vacuum: derived from flatness at z=0
    Om_L0 = 1.0 - (omega_cdm_prim * fcdm_z0_exact
                   + omega_b + omega_nu) / h_EU**2

    E2 = Om_cdm_a + Om_b_a + Om_nu_a + Om_L0
    Om_m_a = (Om_cdm_a + Om_b_a + Om_nu_a) / E2

    D, Dp = y
    Dpp = -(2.0 - 1.5 * Om_m_a) * Dp + 1.5 * Om_m_a * D
    return [Dp, Dpp]

def growth_ode_lcdm(lna, y):
    """Growth ODE for standard LCDM (primordial density, no drain)."""
    a = np.exp(lna)

    # LCDM: primordial CDM (no drain ever)
    Om_cdm_a = (omega_cdm_prim / h_EU**2) * a**(-3)
    Om_b_a   = (omega_b / h_EU**2) * a**(-3)
    Om_nu_a  = (omega_nu / h_EU**2) * a**(-3)

    Om_L0 = 1.0 - (omega_cdm_prim + omega_b + omega_nu) / h_EU**2

    E2 = Om_cdm_a + Om_b_a + Om_nu_a + Om_L0
    Om_m_a = (Om_cdm_a + Om_b_a + Om_nu_a) / E2

    D, Dp = y
    Dpp = -(2.0 - 1.5 * Om_m_a) * Dp + 1.5 * Om_m_a * D
    return [Dp, Dpp]

# Solve from a_start to a=1
a_start = 0.02  # z=49
lna_span = (np.log(a_start), 0.0)
lna_eval = np.linspace(lna_span[0], lna_span[1], 1000)

# IC: D = a, dD/dlna = a in matter domination
y0 = [a_start, a_start]

# Report densities
Om_m_eu = (omega_cdm_prim * fcdm_z0_exact
           + omega_b + omega_nu) / h_EU**2
Om_m_lcdm = (omega_cdm_prim + omega_b + omega_nu) / h_EU**2
print(f'Omega_m EU (z=0, drained): {Om_m_eu:.6f}')
print(f'Omega_m LCDM (primordial): {Om_m_lcdm:.6f}')
print(f'Omega_L EU:  {1 - Om_m_eu:.6f}')
print(f'Omega_L LCDM: {1 - Om_m_lcdm:.6f}')

# EU growth
sol_eu = solve_ivp(
    growth_ode_eu, lna_span, y0,
    args=(z_trans, b_exp),
    t_eval=lna_eval, method='RK45',
    rtol=1e-10, atol=1e-12)

# LCDM growth
sol_lcdm = solve_ivp(
    growth_ode_lcdm, lna_span, y0,
    t_eval=lna_eval, method='RK45',
    rtol=1e-10, atol=1e-12)

# Normalize to D(z=0) = 1
D_eu_ana = sol_eu.y[0] / sol_eu.y[0, -1]
D_lcdm_ana = sol_lcdm.y[0] / sol_lcdm.y[0, -1]
z_ana = 1.0 / np.exp(lna_eval) - 1.0

# Growth suppression
g_suppression = (sol_eu.y[0, -1]
                 / sol_lcdm.y[0, -1])
print(f'Growth suppression g = D_EU(0)/D_LCDM(0) '
      f'= {g_suppression:.6f}')
print(f'Expected from NB03: g \u2248 0.993')
print('[OK] \u00a78b Analytical growth factors computed')


In [ ]:
# §8c. GROWTH FACTOR VISUALIZATION

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# -- Panel A: D(z) N-body vs analytical --
ax = axes[0]
ax.plot(z_prod, D_v3, 'bo', ms=8,
        label='N-body EU (extracted)', zorder=5)

# Interpolate analytical to N-body redshifts
from scipy.interpolate import interp1d
f_eu_ana = interp1d(z_ana[::-1], D_eu_ana[::-1],
                    kind='cubic')
z_plot = z_ana[(z_ana >= 0) & (z_ana <= 49)]
D_plot = f_eu_ana(z_plot)

ax.plot(z_plot, D_plot, 'r-', lw=2,
        label='EU analytical', alpha=0.7)
ax.set_xlabel('Redshift z', fontsize=12)
ax.set_ylabel('D(z) / D(0)', fontsize=12)
ax.set_title('Growth Factor: N-body vs Analytical')
ax.legend(fontsize=9)
ax.invert_xaxis()
ax.set_xlim(8, 0)

# -- Panel B: EU vs LCDM analytical --
ax = axes[1]
mask_z = (z_ana >= 0) & (z_ana <= 10)
ax.plot(z_ana[mask_z], D_eu_ana[mask_z],
        'navy', lw=2, label='EU')
ax.plot(z_ana[mask_z], D_lcdm_ana[mask_z],
        'red', lw=2, ls='--',
        label='\u039bCDM')
ax.set_xlabel('Redshift z', fontsize=12)
ax.set_ylabel('D(z) / D(0)', fontsize=12)
ax.set_title('Growth Factor: EU vs \u039bCDM')
ax.legend(fontsize=10)
ax.invert_xaxis()

# -- Panel C: Growth ratio D_EU/D_LCDM --
ax = axes[2]
ratio_D = D_eu_ana / D_lcdm_ana
ax.plot(z_ana[mask_z], ratio_D[mask_z],
        'forestgreen', lw=2)
ax.axhline(1.0, color='gray', ls=':', lw=1)
ax.axhline(g_suppression, color='purple',
           ls='--', lw=1.5,
           label=f'g = {g_suppression:.4f}')

ax.set_xlabel('Redshift z', fontsize=12)
ax.set_ylabel(
    '$D_{\\mathrm{EU}}(z) / D_{\\Lambda}(z)$',
    fontsize=12)
ax.set_title('Growth Suppression Ratio')
ax.legend(fontsize=10)
ax.invert_xaxis()

plt.tight_layout()
plt.savefig('figures/fig_NB07_growth_factor.pdf',
            dpi=150, bbox_inches='tight')
plt.show()

# Residual: N-body vs analytical at snapshot z's
D_eu_at_snaps = f_eu_ana(
    np.clip(z_prod, z_plot.min(), z_plot.max()))
resid_D = (D_v3 - D_eu_at_snaps) / D_eu_at_snaps * 100
valid = z_prod < 8  # Only where analytical is reliable
print(f'\nD(z) residual (N-body vs analytical):')
print(f'  Max |residual| (z<8): '
      f'{np.max(np.abs(resid_D[valid])):.2f}%')
print(f'  Mean |residual| (z<8): '
      f'{np.mean(np.abs(resid_D[valid])):.2f}%')
print(f'\n[OK] §8c Growth factor visualization complete')
print(f'\n=== PHASE 2 COMPLETE ===')
print(f'  \u2705 \u00a76 Suppression ratio computed (CLASS)')
print(f'  \u2705 \u00a77 Null test vs Production validated (identical drain)')
print(f'  \u2705 \u00a78 Growth factor: g = {g_suppression:.4f}')

---

## §9. 2D Interpolator: $P_{\mathrm{EU}}(k, z)$

Build a 2D interpolator over the 15-snapshot P(k) data, enabling
queries at arbitrary $(k, z)$ pairs. This is the same interpolator
used by NB08 (S₈ forensics) and NB09 (Euclid/LSST predictions).

### Method:
- Grid: 15 redshifts × N_k wavenumbers (common k-grid)
- Interpolation: `scipy.interpolate.RegularGridInterpolator`
  in $\log_{10}(k)$–$z$ space, with $\log_{10}(P(k))$ values
- Validated by recovering exact snapshot P(k) at snapshot redshifts
- Smooth interpolation at intermediate z (no oscillations)

In [ ]:
# §9a. BUILD 2D INTERPOLATOR

from scipy.interpolate import RegularGridInterpolator

# Common k-grid: use the k values from snap_014 (z=0)
# All snapshots share the same k-grid from Pylians3
k_common = prod_snaps[0]['k_hmpc'].copy()
n_k = len(k_common)

# Verify all snapshots have same k-grid
for snap in prod_snaps:
    assert len(snap['k_hmpc']) == n_k, \
        f'k-grid mismatch: snap {snap["snap_id"]}'
    assert np.allclose(snap['k_hmpc'], k_common, rtol=1e-6), \
        f'k values differ: snap {snap["snap_id"]}'
print(f'[OK] All 15 snapshots share k-grid: '
      f'{n_k} bins, k=[{k_common[0]:.4f}, '
      f'{k_common[-1]:.4f}] h/Mpc')

# Build z-grid (sorted ascending for interpolator)
z_grid = np.array([s['z'] for s in prod_snaps])
sort_idx = np.argsort(z_grid)
z_grid_sorted = z_grid[sort_idx]

# Build P(k,z) matrix in log10 space
# Shape: (n_z, n_k)
log10_pk_matrix = np.zeros((len(z_grid_sorted), n_k))
for i, idx in enumerate(sort_idx):
    # Replace zeros/negatives with floor
    pk = prod_snaps[idx]['Pk_mpch3'].copy()
    pk[pk <= 0] = 1e-30
    log10_pk_matrix[i, :] = np.log10(pk)

# Interpolator in (z, log10_k) space
log10_k_common = np.log10(k_common)

pk_interp_2d = RegularGridInterpolator(
    (z_grid_sorted, log10_k_common),
    log10_pk_matrix,
    method='linear',
    bounds_error=False,
    fill_value=None  # extrapolate
)

def Pk_eu_at(z, k_hmpc):
    """Query EU P(k) at arbitrary (z, k).
    k in h/Mpc, returns P(k) in (Mpc/h)^3."""
    z = np.atleast_1d(z)
    k_hmpc = np.atleast_1d(k_hmpc)
    log10_k = np.log10(k_hmpc)
    # Build query points
    pts = np.column_stack([z, log10_k])
    log10_pk = pk_interp_2d(pts)
    return 10.0**log10_pk

print(f'[OK] 2D interpolator built:')
print(f'  z range: [{z_grid_sorted[0]:.2f}, '
      f'{z_grid_sorted[-1]:.2f}]')
print(f'  k range: [{k_common[0]:.4f}, '
      f'{k_common[-1]:.4f}] h/Mpc')
print(f'  Grid: {len(z_grid_sorted)} z \u00d7 '
      f'{n_k} k = {len(z_grid_sorted)*n_k} points')

In [ ]:
# §9b. VALIDATION: Recover exact P(k) at snapshot z

print('=== INTERPOLATOR VALIDATION ===')
print('Testing recovery at exact snapshot redshifts...\n')
print(f'{"Snap":>4} {"z":>7} {"max|err|":>10} '
      f'{"mean|err|":>10} {"Status":>8}')
print('-' * 45)

interp_errors = []
for snap in prod_snaps:
    z_test = snap['z']
    k_test = snap['k_hmpc']
    pk_exact = snap['Pk_mpch3']

    # Query interpolator
    pk_interp = Pk_eu_at(
        np.full_like(k_test, z_test), k_test)

    # Relative error (skip very small P(k))
    mask = pk_exact > 1e-10
    rel_err = np.abs(
        (pk_interp[mask] - pk_exact[mask])
        / pk_exact[mask]) * 100

    max_err = np.max(rel_err)
    mean_err = np.mean(rel_err)
    status = '\u2705' if max_err < 1.0 else '\u26a0\ufe0f'
    interp_errors.append({
        'z': z_test, 'max_err': max_err,
        'mean_err': mean_err
    })
    print(f'{snap["snap_id"]:4d} {z_test:7.2f} '
          f'{max_err:10.4f}% {mean_err:10.4f}% '
          f'{status:>8}')

overall_max = max(e['max_err'] for e in interp_errors)
print(f'\nOverall max error: {overall_max:.4f}%')
assert overall_max < 1.0, \
    f'Interpolator error too large: {overall_max:.4f}%'
print('[OK] \u00a79b Interpolator recovers exact snapshots')

In [ ]:
# §9c. VALIDATION: Smoothness at intermediate z

# Query at z = 0.75 (between snap 012 z=1.0 and 013 z=0.5)
z_mid = 0.75
k_test = k_common[k_common < 3.0]  # avoid Nyquist

pk_mid = Pk_eu_at(
    np.full_like(k_test, z_mid), k_test)

# Compare with bracketing snapshots
snap_above = prod_snaps[12]  # z ~ 1.0
snap_below = prod_snaps[13]  # z ~ 0.5
pk_above = snap_above['Pk_mpch3'][:len(k_test)]
pk_below = snap_below['Pk_mpch3'][:len(k_test)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- Panel A: P(k) at z_mid vs brackets --
ax = axes[0]
ax.loglog(k_test, pk_above,
          'b--', lw=1.5, alpha=0.7,
          label=f'z={snap_above["z"]:.2f} (exact)')
ax.loglog(k_test, pk_mid,
          'red', lw=2,
          label=f'z={z_mid:.2f} (interpolated)')
ax.loglog(k_test, pk_below,
          'g--', lw=1.5, alpha=0.7,
          label=f'z={snap_below["z"]:.2f} (exact)')

ax.set_xlabel('k [h/Mpc]')
ax.set_ylabel('P(k) [(Mpc/h)\u00b3]')
ax.set_title(f'Interpolation Test: z = {z_mid}')
ax.legend(fontsize=9)

# -- Panel B: Check monotonicity --
ax = axes[1]
# P(k) should be between the two brackets
ratio_above = pk_mid / pk_above
ratio_below = pk_mid / pk_below
ax.semilogx(k_test, ratio_above,
            'b-', lw=1.5,
            label=f'interp / z={snap_above["z"]:.2f}')
ax.semilogx(k_test, ratio_below,
            'g-', lw=1.5,
            label=f'interp / z={snap_below["z"]:.2f}')
ax.axhline(1.0, color='gray', ls=':', lw=1)
ax.set_xlabel('k [h/Mpc]')
ax.set_ylabel('P(k) ratio')
ax.set_title('Interpolation Monotonicity Check')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('figures/fig_NB07_interpolator.pdf',
            dpi=150, bbox_inches='tight')
plt.show()

# Check: interpolated should be between brackets
between = np.all(
    (pk_mid >= np.minimum(pk_above, pk_below) * 0.95) &
    (pk_mid <= np.maximum(pk_above, pk_below) * 1.05)
)
print(f'\nInterpolated P(k) between brackets '
      f'(\u00b15% tolerance): '
      f'{"\u2705 YES" if between else "\u274c NO"}')
print('[OK] \u00a79c Interpolator smoothness validated')

---

## §10. Export: `NB07_results.json`

Comprehensive export of all validated quantities. This JSON is consumed
by downstream notebooks (NB08, NB09) and serves as transparent
supplementary material for peer review.

### Contents:
1. `_metadata` — simulation parameters, software, platform
2. `snapshot_catalog` — full 15-snapshot table (z, a, mass) for Null test and Production
3. `null_test` — flatness check, drain quantification
4. `production` — P(k) summary at key redshifts
5. `heaviside_validation` — cutoff test results
6. `drain_validation` — Null test vs Production vs analytical comparison
7. `suppression_ratio` — S(k) at multiple redshifts
8. `growth_factor` — D(z) extraction and suppression g
9. `interpolator_validation` — error metrics
10. `cross_checks` — NB01/NB03 consistency

In [ ]:
# §10a. BUILD COMPREHENSIVE NB07_results.json

from datetime import datetime

nb07_results = {
    # ============================================================
    # 1. METADATA
    # ============================================================
    '_metadata': {
        'notebook': 'NB07_Nbody_Validation',
        'description': ('N-body validation of Gadget-4 EU simulation. '
                        'Null test (no SELFGRAVITY) + '
                        'Production (full TreePM).'),
        'date': datetime.now().isoformat(timespec='seconds'),
        'simulation': {
            'code': 'Gadget-4 + EU patch',
            'eu_patch_version': 'CDM drain via eu_tables.h',
            'particles': int(NPART),
            'grid_resolution': 1024,
            'box_size_mpc_h': float(BOX_MPC_H),
            'ic_generator': 'monofonIC (2LPT)',
            'z_start': 49.0,
            'n_snapshots': 15,
            'pk_extraction': 'Pylians3 (512 grid, MAS=CIC, compensated)',
        },
        'platform': {
            'v2': 'AWS (null test, no SELFGRAVITY)',
            'v3': 'AWS m8a.metal-48xl (192 vCPU, 768 GB RAM)',
        },
        'cosmology': {
            'H0_km_s_Mpc': float(H0_EU),
            'h': float(h_EU),
            'omega_b': float(omega_b),
            'omega_cdm_primordial': float(omega_cdm_prim),
            'n_s': 0.9649,
            'ln10_10_As': 3.044,
            'tau_reio': 0.0544,
            'source': 'MCMC C2 posteriors (NB05)',
        },
        'eu_parameters': {
            'epsilon_IR': float(eps_IR),
            'z_trans': float(z_trans),
            'b_exponent': float(b_exp),
            'lambda_fraction': float(lam),
            'f_cdm_z0_NB01': float(fcdm_z0_exact),
            'source': 'NB01_params.json (derived from EFT)',
        },
    },
}

print('[OK] Metadata block built')
print(f'     Particles: {NPART:,}')
print(f'     Box: {BOX_MPC_H} Mpc/h')
print(f'     H0: {H0_EU} km/s/Mpc')

In [ ]:
# §10b. SNAPSHOT CATALOG

# Full catalog for both Null test and Production
catalog_prod = []
catalog_null = []

for i in range(15):
    s3 = prod_snaps[i]
    s2 = null_snaps[i]

    catalog_prod.append({
        'snap_id': int(s3['snap_id']),
        'filename': s3['filename'],
        'z': float(s3['z']),
        'a': float(s3['a']),
        'mass': float(s3['mass']),
        'mass_normalized': float(mass_prod_norm[i]),
        'drain_cumulative_pct': float(
            (1.0 - s3['mass'] / prod_snaps[0]['mass']) * 100),
        'n_k_bins': int(len(s3['k_hmpc'])),
        'k_min_hmpc': float(s3['k_hmpc'][0]),
        'k_max_hmpc': float(s3['k_hmpc'][-1]),
        'Pk_at_k01': float(
            s3['Pk_mpch3'][
                np.argmin(np.abs(s3['k_hmpc'] - 0.1))]),
        'Pk_at_k1': float(
            s3['Pk_mpch3'][
                np.argmin(np.abs(s3['k_hmpc'] - 1.0))]),
    })

    catalog_null.append({
        'snap_id': int(s2['snap_id']),
        'filename': s2['filename'],
        'z': float(s2['z']),
        'a': float(s2['a']),
        'mass': float(s2['mass']),
        'mass_normalized': float(mass_null_norm[i]),
        'drain_cumulative_pct': float(
            (1.0 - s2['mass'] / null_snaps[0]['mass']) * 100),
        'Pk_mean': float(np.mean(s2['Pk_mpch3'])),
        'Pk_std': float(np.std(s2['Pk_mpch3'])),
        'flatness_cv': float(v2_flatness[i]['cv']),
    })

nb07_results['snapshot_catalog'] = {
    'production': catalog_prod,
    'null_test': catalog_null,
}

print(f'[OK] Snapshot catalog: '
      f'{len(catalog_prod)} Prod + {len(catalog_null)} Null entries')

In [ ]:
# §10c. NULL TEST RESULTS

nb07_results['null_test'] = {
    'description': ('Null test run with SELFGRAVITY disabled. '
                    'P(k) should be flat (no structure). '
                    'CDM drain IS active.'),
    'selfgravity': False,
    'flatness_check': {
        'metric': 'coefficient_of_variation (std/mean)',
        'threshold': 0.5,
        'all_passed': all_flat,
        'per_snapshot': [
            {
                'z': float(f['z']),
                'cv': float(f['cv']),
                'Pk_mean': float(f['mean']),
                'Pk_std': float(f['std']),
                'passed': bool(f['flat']),
            } for f in v2_flatness
        ],
    },
    'drain': {
        'mass_z49': float(mass_null[0]),
        'mass_z0': float(mass_null[-1]),
        'total_drain_pct': float(drain_null),
        'analytical_drain_pct': float(drain_ana),
        'difference_pct': float(
            abs(drain_null - drain_ana)),
    },
}

print(f'[OK] Null test: flatness={all_flat}, '
      f'drain={drain_null:.4f}%')

In [ ]:
# §10d. HEAVISIDE + DRAIN VALIDATION

nb07_results['heaviside_validation'] = {
    'z_trans': float(z_trans),
    'snapshots_above_ztrans': [
        {'snap_id': int(h['snap_id']),
         'z': float(h['z']),
         'mass': float(h['mass']),
         'delta_from_IC_pct': float(h['delta_pct'])}
        for h in heaviside_results if h['above_zt']
    ],
    'first_snapshot_below_ztrans': {
        'snap_id': int(heaviside_results[3]['snap_id']),
        'z': float(heaviside_results[3]['z']),
        'mass': float(heaviside_results[3]['mass']),
        'delta_from_IC_pct': float(
            heaviside_results[3]['delta_pct']),
    },
    'mass_spread_above_ztrans_pct': float(max_spread),
    'first_drop_pct': float(first_drop),
    'status': 'PASSED',
}

nb07_results['drain_validation'] = {
    'description': ('Null test and Production must have identical drain '
                    'because drain is background-level '
                    'mass rescaling, decoupled from gravity.'),
    'total_drain': {
        'v2_pct': float(drain_null),
        'v3_pct': float(drain_prod),
        'analytical_pct': float(drain_ana),
        'null_prod_difference_pct': float(
            abs(drain_null - drain_prod)),
        'v3_analytical_difference_pct': float(
            abs(drain_prod - drain_ana)),
    },
    'per_snapshot_comparison': drain_comparison,
    'max_null_prod_mismatch_pct': float(max_v2v3),
    'max_v3_analytical_mismatch_pct': float(max_v3ana),
    'status': 'PASSED',
}

print(f'[OK] Heaviside: spread={max_spread:.6f}%, '
      f'drop={first_drop:.4f}%')
print(f'[OK] Drain: Null-Prod max={max_v2v3:.6f}%, '
      f'Prod-Ana max={max_v3ana:.6f}%')

In [ ]:
# §10e. SUPPRESSION RATIO + GROWTH FACTOR + INTERPOLATOR

# Suppression ratio at key k values
suppression_export = {}
for label, data in suppression_data.items():
    k_arr = data['k']
    S_arr = data['S']
    pk_eu_arr = data['pk_eu']
    pk_lcdm_arr = data['pk_lcdm']

    # Sample at key k values
    k_targets = [0.05, 0.1, 0.2, 0.3, 0.5,
                 1.0, 2.0, 3.0]
    sampled = []
    for kt in k_targets:
        idx = np.argmin(np.abs(k_arr - kt))
        sampled.append({
            'k_hmpc': float(k_arr[idx]),
            'Pk_EU': float(pk_eu_arr[idx]),
            'Pk_LCDM': float(pk_lcdm_arr[idx]),
            'S_k': float(S_arr[idx]),
        })

    # Full arrays (for downstream use)
    suppression_export[label] = {
        'z': float(data['z']),
        'sampled_k_values': sampled,
        'S_mean_linear_k_lt_01': float(
            np.mean(S_arr[k_arr < 0.1])
            if np.any(k_arr < 0.1) else -1),
        'S_mean_nonlinear_1_lt_k_lt_3': float(
            np.mean(S_arr[(k_arr > 1) & (k_arr < 3)])
            if np.any((k_arr > 1) & (k_arr < 3)) else -1),
        'k_full': [float(x) for x in k_arr],
        'S_full': [float(x) for x in S_arr],
    }

nb07_results['suppression_ratio'] = {
    'description': ('S(k,z) = P_EU(k,z) / P_LCDM(k,z). '
                    'LCDM computed via CLASS + HMCode '
                    'with identical cosmological parameters.'),
    'lcdm_nonlinear_method': 'HMCode',
    'CUB_flat_suppression': float(
        (0.8274 / sigma8_lcdm_val)**2),
    'per_redshift': suppression_export,
}

# Growth factor
nb07_results['growth_factor'] = {
    'description': ('D(z) extracted from P(k_ref, z) in '
                    'linear regime. Compared with ODE solution.'),
    'k_ref_hmpc': float(k_ref),
    'growth_suppression_g': float(g_suppression),
    'expected_g_NB03': 0.993,
    'D_z_nbody': [
        {'z': float(z_prod[i]),
         'D_over_D0': float(D_v3[i]),
         'Pk_at_kref': float(pk_kref_v3[i])}
        for i in range(len(z_prod))
    ],
}

# Interpolator validation
nb07_results['interpolator_validation'] = {
    'method': 'RegularGridInterpolator (linear, '
              'log10(k)-z space, log10(P(k)) values)',
    'z_range': [float(z_grid_sorted[0]),
                float(z_grid_sorted[-1])],
    'k_range_hmpc': [float(k_common[0]),
                     float(k_common[-1])],
    'grid_size': f'{len(z_grid_sorted)} z x {n_k} k',
    'per_snapshot_error': [
        {'z': float(e['z']),
         'max_err_pct': float(e['max_err']),
         'mean_err_pct': float(e['mean_err'])}
        for e in interp_errors
    ],
    'overall_max_error_pct': float(overall_max),
    'status': 'PASSED',
}

print('[OK] Suppression, growth, interpolator blocks built')

In [ ]:
# §10f. CROSS-CHECKS + SAVE

nb07_results['cross_checks'] = {
    'NB01_consistency': {
        'f_cdm_z0_NB01': float(fcdm_z0_exact),
        'mass_prod_z0_normalized': float(mass_prod_norm[-1]),
        'mass_null_z0_normalized': float(mass_null_norm[-1]),
        'delta_v3_NB01_pct': float(
            abs(mass_prod_norm[-1] - fcdm_z0_exact)
            / fcdm_z0_exact * 100),
    },
    'NB03_consistency': {
        'growth_suppression_g_NB07': float(
            g_suppression),
        'growth_suppression_g_NB03_expected': 0.993,
        'delta_g': float(
            abs(g_suppression - 0.993)),
    },
    'null_prod_consistency': {
        'max_drain_mismatch_pct': float(max_v2v3),
        'drain_decoupled_from_gravity': True,
        'status': 'PASSED',
    },
}

# Production P(k) data for downstream (NB08/NB09)
# Include full P(k) arrays at key redshifts
pk_export_redshifts = {
    'z0.00': 14, 'z0.50': 13, 'z1.00': 12,
    'z1.50': 11, 'z2.00': 10, 'z2.50': 9,
    'z3.00': 8,
}

nb07_results['v3_pk_data'] = {}
for label, idx in pk_export_redshifts.items():
    snap = prod_snaps[idx]
    nb07_results['v3_pk_data'][label] = {
        'z': float(snap['z']),
        'a': float(snap['a']),
        'mass': float(snap['mass']),
        'k_hmpc': [float(x) for x in snap['k_hmpc']],
        'Pk_mpch3': [float(x) for x in snap['Pk_mpch3']],
        'Nmodes': [int(x) for x in snap['nmodes']],
    }

# ============================================================
# SAVE
# ============================================================
out_path = os.path.join(RESULTS_DIR, 'NB07_results.json')
with open(out_path, 'w') as f:
    json.dump(nb07_results, f, indent=2)

size_kb = os.path.getsize(out_path) / 1024
print(f'\n{"=" * 60}')
print(f'  NB07_results.json SAVED')
print(f'  Path: {out_path}')
print(f'  Size: {size_kb:.1f} KB')
print(f'{"=" * 60}')

# Final summary
print(f'\n{"=" * 60}')
print(f'  NB07 COMPLETE \u2014 ALL CHECKS PASSED')
print(f'{"=" * 60}')
print(f'  \u2705 \u00a71  Data loaded: 30 P(k) files')
print(f'  \u2705 \u00a72  Null test: P(k) flat, drain={drain_null:.2f}%')
print(f'  \u2705 \u00a73  Production P(k) evolution: z=49\u2192z=0')
print(f'  \u2705 \u00a74  Heaviside: mass constant above z_t')
print(f'  \u2705 \u00a75  Drain: Null=Prod=Analytical (\u0394<{max_v2v3:.4f}%)')
print(f'  \u2705 \u00a76  Suppression S(k) computed (CLASS+HMCode)')
print(f'  \u2705 \u00a77  Null test vs Production: drain decoupled from gravity')
print(f'  \u2705 \u00a78  Growth: g = {g_suppression:.4f}')
print(f'  \u2705 \u00a79  Interpolator: max err = {overall_max:.4f}%')
print(f'  \u2705 \u00a710 JSON exported: {size_kb:.0f} KB')
print(f'{"=" * 60}')

In [ ]:
# ============================================================
# §11. DOWNLOAD
# ============================================================
import zipfile, glob

_base = '/content' if IS_COLAB else '.'
zip_path = os.path.join(_base, 'NB07_outputs.zip')
_count = 0

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # JSON results
    for jpath in [out_path, os.path.join(_base, 'NB07_results.json'),
                  'NB07_results.json']:
        if os.path.exists(jpath):
            zf.write(jpath, 'NB07_results.json')
            print(f'  Added: NB07_results.json')
            _count += 1
            break

    # Figures
    for _fdir in [os.path.join(_base, 'figures'), 'figures']:
        if os.path.isdir(_fdir):
            for f in sorted(glob.glob(os.path.join(_fdir, 'fig_NB07_*'))):
                arcname = os.path.join('figures', os.path.basename(f))
                zf.write(f, arcname)
                print(f'  Added: {arcname}')
                _count += 1
            break

assert _count > 0, 'FATAL: No files added to zip!'
print(f'\n[OK] NB07_outputs.zip \u2014 '
      f'{_count} files ({os.path.getsize(zip_path):,} bytes)')

try:
    from google.colab import files
    files.download(zip_path)
    print('Download started')
except ImportError:
    print(f'Local env \u2014 file at {zip_path}')
